# Analisis avanzado del dataset Titanic

En este análisis se utilizarán técnicas de sumarización, agrupación, agregación y análisis de dispersión para estudiar diferentes características de los pasajeros del Titanic. El objetivo es encontrar patrones relacionados con la supervivencia, analizar posibles valores atípicos y comprender cómo estos aspectos pueden afectar a futuros modelos de Machine Learning.


In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

df.head()

In [ ]:
survival_rate = df['Survived'].value_counts(normalize=True) * 100
survival_rate

## Pregunta A — Sumarización categórica

la tasa de supervivencia fue de aproximadamente 38.38%. esto significa que menos de la mitad de los pasajeros sobrevivio.



In [ ]:
survival_by_sex = df.groupby('Sex')['Survived'].mean() * 100
survival_by_sex

### pregunta B - agrupacion y agregacion

aproximadamente 74.20% de las mujeres sobrevivio, mientras que solo 18.89% de los hombres. esto muestra una diferencia bastante grande entre los dos grupos.


In [ ]:
quartiles = df['Fare'].quantile([0.25, 0.75])
quartiles

In [ ]:
q1 = quartiles[0.25]
q3 = quartiles[0.75]

iqr = q3 - q1

print("q1:", q1)
print("q3:", q3)
print("iqr:", iqr)

In [ ]:
upper_limit = q3 + 1.5 * iqr

print("limite superior:", upper_limit)

In [ ]:
outliers = df[df['Fare'] > upper_limit]

print("cantidad de outliers:", len(outliers))
outliers

In [ ]:
outliers['Pclass'].value_counts()

### pregunta C - outliers en fare

las tarifas que estan por encima del limite son consideradas outliers. la mayoria de estos pasajeros pertenecia a primera clase. esto no significa necesariamente que sean errores, ya que una tarifa alta puede ser valida.


In [ ]:
fare_mean = df['Fare'].mean()
fare_median = df['Fare'].median()

print("media:", fare_mean)
print("mediana:", fare_median)

### pregunta D - media y mediana

la media es mucho mayor que la mediana porque hay algunas tarifas muy altas que suben el promedio. en knn, si no escalamos la variable, estos valores pueden tener demasiado peso en las distancias y afectar el modelo.


In [ ]:
sample_size = 150

survived_count = round(df['Survived'].mean() * sample_size)
not_survived_count = sample_size - survived_count

survived = df[df['Survived'] == 1].sample(
    n=survived_count,
    random_state=42
)

not_survived = df[df['Survived'] == 0].sample(
    n=not_survived_count,
    random_state=42
)

sample = pd.concat([survived, not_survived])

sample = sample.sample(frac=1, random_state=42).reset_index(drop=True)

sample['Survived'].value_counts()

In [ ]:
print("tamano de la muestra:", len(sample))

print("\nproporcion de la muestra:")
print(sample['Survived'].value_counts(normalize=True))

print("\nproporcion original:")
print(df['Survived'].value_counts(normalize=True))

### pregunta E - muestreo proporcional

se mantuvo una proporcion similar a la de la base original. esto evita un sesgo de muestreo y hace que la muestra represente mejor los datos originales.


In [ ]:
df.groupby('Survived')['Age'].mean()

### pregunta F - valores faltantes

se puede introducir un sesgo por los datos faltantes, ya que pandas ignora los nan. si los datos faltantes pertenecen principalmente a un grupo, la media puede no representar correctamente a ese grupo y afectar el modelo.


In [ ]:
outliers[['PassengerId', 'Name', 'Pclass', 'Fare']].sort_values(
    by='Fare',
    ascending=False
)

### pregunta G - outliers

no eliminaria los outliers automaticamente. una tarifa muy alta puede ser un dato real, especialmente si pertenece a primera clase. primero revisaria si realmente es un error.


In [ ]:
df['Titulo'] = df['Name'].str.extract(r',\s*([^.]*)\.', expand=False)

df[['Name', 'Titulo']].head()

In [ ]:
df['Titulo'].value_counts()

In [ ]:
df.groupby('Titulo')['Age'].median()

### pregunta H - imputacion por titulo

usar la mediana por titulo seria mas preciso porque los titulos dan una idea de la edad. por ejemplo, master normalmente corresponde a niños y mr a adultos. asi podemos hacer una mejor estimacion de las edades faltantes.


In [ ]:
survived_variance = df['Survived'].var()

print("varianza:", survived_variance)

### pregunta I - varianza de survived

si la varianza fuera 0, significaria que todos tienen el mismo resultado, todos sobrevivieron o todos murieron. el modelo no tendria dos clases que aprender y no podria hacer una clasificacion util.


In [ ]:
grouped = df.groupby(
    ['Pclass', 'Sex', 'Embarked']
)['PassengerId'].count()

grouped

In [ ]:
small_groups = grouped[grouped <= 2]
small_groups

### pregunta J - grupos pequeños

ocurriria sobreajuste porque el modelo estaria aprendiendo reglas de grupos demasiado pequeños. esto puede hacer que funcione bien con los datos de entrenamiento pero mal con datos nuevos.


### conclusion

en este analisis encontramos diferencias importantes en la supervivencia, outliers en las tarifas y datos faltantes en la edad. tambien vimos que es importante mantener una buena proporcion en la muestra y evitar grupos demasiado pequeños para no causar sobreajuste.
